In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv
/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/LCDataDictionary.xlsx


In [2]:
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
data = pd.read_csv('/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv')
data.head()

/tmp/ipykernel_16/4254957762.py:1: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv')


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## Target Variable

In [4]:
lgd_data = data[data['loan_status'] == 'Charged Off'].copy()

# LGD = 1 - (Recoveries / Funded Amount)
lgd_data['lgd_target'] = 1 - (lgd_data['recoveries'] / lgd_data['funded_amnt'])

# Cap the LGD between 0.0 (no loss) and 1.0 (total loss) to fix any weird data anomalies
lgd_data['lgd_target'] = lgd_data['lgd_target'].clip(lower=0.0, upper=1.0)

print(f"LGD dataset shape before cleaning: {lgd_data.shape}")

LGD dataset shape before cleaning: (261655, 146)


# Data Cleaning

## Numerical Cleaning
1. **Dropping NaN(s)**: Drop columns with almost every element as NaN.
2. **Dropping Cheater Features**: Drop columns that causes leakages in our pipeline.
3. **Fill in remaining**: Fill remaining NaNs with some values for modeling.

### Drop NaNs

In [5]:
print("--- 3. Cleaning Data ---")

# A. Drop High-NaN Columns (>30% missing)
threshold = 0.30 
lgd_data = lgd_data.dropna(thresh=len(lgd_data) * (1 - threshold), axis=1)

--- 3. Cleaning Data ---


### Drop Cheaters

In [6]:
# B. Drop Cheater Columns (CRUCIAL: 'recoveries' must be dropped NOW so it doesn't leak into training)
leakage_cols = [
    'loan_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt',
    'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'debt_settlement_flag', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage',
    'settlement_term', 'hardship_flag', 'hardship_type', 
    'hardship_reason', 'hardship_status', 'deferphal_term',
    'payment_plan_start_date', 'orig_projected_additional_accrued_interest'
]
cols_to_drop = [col for col in leakage_cols if col in lgd_data.columns]
lgd_data = lgd_data.drop(columns=cols_to_drop)

### Fill Remaining

In [7]:
# C. Fill Remaining NaNs
num_cols = lgd_data.select_dtypes(include=['float64', 'int64']).columns
lgd_data[num_cols] = lgd_data[num_cols].fillna(lgd_data[num_cols].median())

cat_cols = lgd_data.select_dtypes(include=['object']).columns
lgd_data[cat_cols] = lgd_data[cat_cols].fillna('Unknown')

## Categorical Encoding

In [8]:
# ---------------------------------------------------------
# STEP 4: Format and Encode Categorical Variables
# ---------------------------------------------------------
print("--- 4. Encoding Categorical Data ---")

if 'term' in lgd_data.columns:
    lgd_data['term'] = lgd_data['term'].str.extract(r'(\d+)').astype(float)

if 'emp_length' in lgd_data.columns:
    emp_map = {
        '10+ years': 10, '9 years': 9, '8 years': 8, '7 years': 7,
        '6 years': 6, '5 years': 5, '4 years': 4, '3 years': 3,
        '2 years': 2, '1 year': 1, '< 1 year': 0, 'Unknown': 0
    }
    lgd_data['emp_length'] = lgd_data['emp_length'].map(emp_map).fillna(0)

# Drop useless and high-cardinality columns
useless_text_cols = ['emp_title', 'title', 'zip_code', 'url', 'desc']
lgd_data = lgd_data.drop(columns=[col for col in useless_text_cols if col in lgd_data.columns])

categorical_cols = lgd_data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if lgd_data[col].nunique() > 40:
        lgd_data = lgd_data.drop(columns=[col])

# Label Encode
le = LabelEncoder()
categorical_cols = lgd_data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    lgd_data[col] = le.fit_transform(lgd_data[col].astype(str))

print(f"Final LGD dataset shape: {lgd_data.shape}")

--- 4. Encoding Categorical Data ---
Final LGD dataset shape: (261655, 67)


In [9]:
print("----Final Dataset for Modeling----")
lgd_data.head()

----Final Dataset for Modeling----


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,lgd_target
5040,8000,8000,8000.0,36.0,6.46,245.05,0,0,10,1,...,98.0,25.0,0.0,0.0,232617.0,20910.0,16700.0,23181.0,1,1.0
25963,6000,6000,6000.0,36.0,14.47,206.44,2,11,0,1,...,80.0,0.0,0.0,0.0,13200.0,5005.0,4100.0,5400.0,0,1.0
41393,10000,10000,10000.0,36.0,8.81,317.12,0,4,0,1,...,100.0,50.0,1.0,0.0,327790.0,30771.0,12000.0,34590.0,0,1.0
55148,10000,10000,10000.0,60.0,27.27,306.97,4,24,10,5,...,91.7,100.0,0.0,0.0,25493.0,23289.0,1800.0,21693.0,0,1.0
58105,35000,35000,35000.0,36.0,16.14,1232.92,2,13,0,1,...,95.5,44.4,1.0,0.0,142518.0,87111.0,20200.0,90118.0,0,1.0


# Model Building

## Train/Test Split

In [10]:
X = lgd_data.drop(columns=['lgd_target'])
y = lgd_data['lgd_target']

# Note: We do NOT stratify here because 'y' is a continuous percentage, not classes (0 or 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## XGBoost Training

In [11]:
print("--- 5. Training LGD XGBoost Regressor ---")
# We use XGBRegressor for continuous percentage prediction
xgb_lgd_model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

xgb_lgd_model.fit(X_train, y_train)

--- 5. Training LGD XGBoost Regressor ---


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [12]:
# ---------------------------------------------------------
# Evaluate Model Performance
# ---------------------------------------------------------
y_pred = xgb_lgd_model.predict(X_test)

# Ensure predictions don't mathematically fall outside 0% to 100%
y_pred = np.clip(y_pred, 0.0, 1.0)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- LGD Model Evaluation ---")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE):      {mae:.4f}")
print(f"R-squared (R2):                 {r2:.4f}")
print(f"Average actual LGD:             {y_test.mean():.4f}")
print(f"Average predicted LGD:          {y_pred.mean():.4f}")


--- LGD Model Evaluation ---
Root Mean Squared Error (RMSE): 0.0933
Mean Absolute Error (MAE):      0.0628
R-squared (R2):                 0.0171
Average actual LGD:             0.9266
Average predicted LGD:          0.9265


In [13]:
# ---------------------------------------------------------
# CUSTOM BUSINESS ACCURACY FOR LGD REGRESSOR
# ---------------------------------------------------------
# Calculate the absolute difference between actual and predicted LGD
absolute_errors = np.abs(y_test - y_pred)

# Define our acceptable margin of error (e.g., +/- 10%)
tolerance_threshold = 0.10

# Count how many predictions fell within that acceptable margin
correct_predictions = (absolute_errors <= tolerance_threshold).sum()
total_predictions = len(y_test)

# Calculate the "Business Accuracy"
business_accuracy = correct_predictions / total_predictions

print(f"LGD Business Accuracy (within +/- 10%): {business_accuracy * 100:.2f}%")

# Let's also check a tighter +/- 5% margin
tight_correct = (absolute_errors <= 0.05).sum()
tight_accuracy = tight_correct / total_predictions
print(f"LGD Strict Accuracy (within +/- 5%):   {tight_accuracy * 100:.2f}%")

LGD Business Accuracy (within +/- 10%): 91.00%
LGD Strict Accuracy (within +/- 5%):   45.16%


In [14]:
# ---------------------------------------------------------
# Exporting the LGD Model
# ---------------------------------------------------------
joblib.dump(xgb_lgd_model, 'lgd_model.pkl')
joblib.dump(X_train.columns.tolist(), 'lgd_features.pkl')

print("\nLGD Model saved as 'lgd_model.pkl'")
print("LGD Feature list saved as 'lgd_features.pkl'")


LGD Model saved as 'lgd_model.pkl'
LGD Feature list saved as 'lgd_features.pkl'
